In [3]:
# Import data

from datasets import load_dataset
ds = load_dataset("ucirvine/sms_spam")

In [4]:
import pandas as pd

df = pd.DataFrame(ds['train'])
print(df.head())

                                                 sms  label
0  Go until jurong point, crazy.. Available only ...      0
1                    Ok lar... Joking wif u oni...\n      0
2  Free entry in 2 a wkly comp to win FA Cup fina...      1
3  U dun say so early hor... U c already then say...      0
4  Nah I don't think he goes to usf, he lives aro...      0


In [5]:
# convert training dataframe to text file

df.to_csv('training_data.txt', sep='\t', index=False)

In [15]:
# Break all words into a list of small words and convert to lower case and remove punctuation

df['words'] = df['sms'].str.lower()
df['words'] = df['words'].str.split()
df['words'] = df['words'].apply(lambda x: [word.strip('.,!?*%^&")') for word in x])

print(df.words)

0       [go, until, jurong, point, crazy, available, o...
1                          [ok, lar, joking, wif, u, oni]
2       [free, entry, in, 2, a, wkly, comp, to, win, f...
3       [u, dun, say, so, early, hor, u, c, already, t...
4       [nah, i, don't, think, he, goes, to, usf, he, ...
                              ...                        
5569    [this, is, the, 2nd, time, we, have, tried, 2,...
5570         [will, ü, b, going, to, esplanade, fr, home]
5571    [pity, , was, in, mood, for, that, so...any, o...
5572    [the, guy, did, some, bitching, but, i, acted,...
5573                     [rofl, its, true, to, its, name]
Name: words, Length: 5574, dtype: object


In [16]:
# define spam and ham dataframes
df_spam = df[df['label'] == 0]
df_ham = df[df['label'] == 1]
print(df_spam.head())

                                                 sms  label  \
0  Go until jurong point, crazy.. Available only ...      0   
1                    Ok lar... Joking wif u oni...\n      0   
3  U dun say so early hor... U c already then say...      0   
4  Nah I don't think he goes to usf, he lives aro...      0   
6  Even my brother is not like to speak with me. ...      0   

                                               words  
0  [go, until, jurong, point, crazy, available, o...  
1                     [ok, lar, joking, wif, u, oni]  
3  [u, dun, say, so, early, hor, u, c, already, t...  
4  [nah, i, don't, think, he, goes, to, usf, he, ...  
6  [even, my, brother, is, not, like, to, speak, ...  


In [17]:
# write each word in the list of words to consist 
words_counts = df['words'].explode().value_counts()
words_counts_spam = df_spam['words'].explode().value_counts()
words_counts_ham = df_ham['words'].explode().value_counts()

print(words_counts)
print(words_counts_spam)
print(words_counts_ham)


# Create a dictionary of words and their counts for spam and ham
words_dic = {}
for word in words_counts.index:
    words_dic[word] = {
        'spam': words_counts_spam.get(word, 0),
        'ham': words_counts_ham.get(word, 0)
    }   

print(words_dic)


words
to             2249
i              2238
you            2123
a              1441
the            1332
               ... 
now1              1
pity              1
so...any          1
suggestions       1
bitching          1
Name: count, Length: 10198, dtype: int64
words
i              2193
you            1836
to             1562
the            1128
a              1063
               ... 
salesman          1
pity              1
so...any          1
suggestions       1
bitching          1
Name: count, Length: 8062, dtype: int64
words
to              687
a               378
call            344
you             287
your            263
               ... 
2.50              1
house             1
£750              1
087187272008      1
now1              1
Name: count, Length: 3184, dtype: int64
{'to': {'spam': np.int64(1562), 'ham': np.int64(687)}, 'i': {'spam': np.int64(2193), 'ham': np.int64(45)}, 'you': {'spam': np.int64(1836), 'ham': np.int64(287)}, 'a': {'spam': np.int64(1063), 'ham': np

In [18]:
# Convert the dictionary to a dataframe
words_df = pd.DataFrame.from_dict(words_dic, orient='index')
print(words_df.head())

     spam  ham
to   1562  687
i    2193   45
you  1836  287
a    1063  378
the  1128  204


In [19]:
# Calculate the total number of spam and ham messages
total_spam = len(df_spam)
total_ham = len(df_ham)

print(f'Total spam messages: {total_spam}')
print(f'Total ham messages: {total_ham}')

Total spam messages: 4827
Total ham messages: 747


In [20]:
# Calculate the probability of each word being spam or ham i.e. P(word|spam) and P(word|ham)
words_df['p_word_spam'] = words_df['spam'] / total_spam
words_df['p_word_ham'] = words_df['ham'] / total_ham

print(words_df.head())

     spam  ham  p_word_spam  p_word_ham
to   1562  687     0.323596    0.919679
i    2193   45     0.454319    0.060241
you  1836  287     0.380360    0.384203
a    1063  378     0.220220    0.506024
the  1128  204     0.233686    0.273092


In [21]:
# Calculate the probability of massaging being spam or ham i.e. P(spam) and P(ham)
p_spam = total_spam / len(df)
p_ham = total_ham / len(df)
print(f'P(spam): {p_spam}')
print(f'P(ham): {p_ham}')

P(spam): 0.8659849300322928
P(ham): 0.1340150699677072


In [23]:
# Calculate P(spam|word) and P(ham|word) using Bayes theorem
import numpy as np
def spam_probability(word, message):
    if word in words_df.index:
        p_word_spam = words_df.loc[word, 'p_word_spam']
        p_word_ham = words_df.loc[word, 'p_word_ham']
    else:
        p_word_spam = 0
        p_word_ham = 0

    p_spam_given_word = (p_word_spam * p_spam) / ((p_word_spam * p_spam) + (p_word_ham * p_ham))
    p_ham_given_word = (p_word_ham * p_ham) / ((p_word_spam * p_spam) + (p_word_ham * p_ham))

    return p_spam_given_word, p_ham_given_word

print(spam_probability('free', 'You have won a free ticket!'))

(np.float64(0.21454545454545457), np.float64(0.7854545454545455))


In [25]:
# Create Class of Naive Bayes Classifier

class NaiveBayes_Classifier:
    def __init__(self):
        self.p_spam = 0
        self.p_ham = 0
        self.words_df = None

    def train(self, df):
        self.df = df
        self.df_spam = df[df['label'] == 0]
        self.df_ham = df[df['label'] == 1]

        self.total_spam = len(self.df_spam)
        self.total_ham = len(self.df_ham)

        self.p_spam = self.total_spam / len(df)
        self.p_ham = self.total_ham / len(df)

        words_counts = df['words'].explode().value_counts()
        words_counts_spam = self.df_spam['words'].explode().value_counts()
        words_counts_ham = self.df_ham['words'].explode().value_counts()

        words_dic = {}
        for word in words_counts.index:
            words_dic[word] = {
                'spam': words_counts_spam.get(word, 0),
                'ham': words_counts_ham.get(word, 0)
            }   

        self.words_df = pd.DataFrame.from_dict(words_dic, orient='index')
        self.words_df['p_word_spam'] = self.words_df['spam'] / self.total_spam
        self.words_df['p_word_ham'] = self.words_df['ham'] / self.total_ham
    def predict(self, message):
        words = message.lower().split()
        p_spam_given_message = self.p_spam
        p_ham_given_message = self.p_ham

        for word in words:
            if word in self.words_df.index:
                p_word_spam = self.words_df.loc[word, 'p_word_spam']
                p_word_ham = self.words_df.loc[word, 'p_word_ham']
            else:
                p_word_spam = 0
                p_word_ham = 0

            p_spam_given_message *= p_word_spam
            p_ham_given_message *= p_word_ham

        return p_spam_given_message, p_ham_given_message 
